# Ex2_SUBNET
Encode the measured history, propagate one latent step, and predict the terminal output.

With `RUN_TRAINING = False`, load the pretrained parameters and display the model. View training history with `tensorboard --logdir runs`.


In [ ]:
from utils import configure_runtime
configure_runtime()
from datetime import datetime
import numpy as np
import torch
from torch.utils.tensorboard import SummaryWriter

from data import load_data
from identification import SUBNET
from utils import ROOT, save_metrics, show_model


## 1. Configuration

In [ ]:
EXPERIMENT = 2
NAME = "Ex2_SUBNET"
RUN_TRAINING = False
SEED = 0
system = dict(nx=3, nu=2, nw=3, N=40, T=0.02,
              E=np.eye(3), C=np.array([[1., 0., 0.], [0., 0., 1.]]),
              integration="euler", rk4_steps=1, m=0.041, l1=0.029, l2=0.033)
REFERENCE_DIR = ROOT / "runs" / NAME
OUTPUT_DIR = ROOT / "runs" / "new" / (NAME + "_" + datetime.now().strftime("%Y%m%d-%H%M%S"))
np.random.seed(SEED)
torch.manual_seed(SEED)
LATENT_DIMENSION = 3
NETWORK_WIDTH = 128
NETWORK_DEPTH = 2
BATCH_SIZE = 256
WEIGHT_DECAY = 3e-5
STAGE1_EPOCHS, STAGE2_EPOCHS = 3000, 2000
STAGE1_LR, STAGE2_LR = 3e-4, 1e-5

system["hidden_limit"] = 2.5


## 2. Data and encoder

In [ ]:
data = load_data(EXPERIMENT)
problem = SUBNET(data, LATENT_DIMENSION, NETWORK_WIDTH, NETWORK_DEPTH, BATCH_SIZE, WEIGHT_DECAY, SEED)


## 3. Normalized training, then raw PEM

In [ ]:
if RUN_TRAINING:
    OUTPUT_DIR.mkdir(parents=True)
    with SummaryWriter(OUTPUT_DIR) as writer:
        stage1 = problem.train_phase("normalized", STAGE1_LR, STAGE1_EPOCHS, writer, OUTPUT_DIR)
        stage2 = problem.train_phase("raw", STAGE2_LR, STAGE2_EPOCHS, writer, OUTPUT_DIR, offset=len(stage1))
    np.savez(OUTPUT_DIR / "history.npz", normalized=stage1, raw=stage2)
else:
    checkpoint = torch.load(REFERENCE_DIR / "model_params_opt.pth", map_location="cpu", weights_only=False)
    problem.load(checkpoint)


## 4. Loaded model

In [ ]:
directory = OUTPUT_DIR if RUN_TRAINING else REFERENCE_DIR
checkpoint = torch.load(directory / "model_params_opt.pth", map_location="cpu", weights_only=False)
problem.load(checkpoint)
if RUN_TRAINING:
    save_metrics(directory, checkpoint["train_pem"], checkpoint["val_pem"], checkpoint["epoch"])
model = torch.nn.ModuleDict({key: getattr(problem.model, key) for key in ("encoder", "fn", "hn")})
model.eval()
show_model(directory, model)


Checkpoint: runs/Ex2_SUBNET/model_params_opt.pth
ModuleDict(
  (encoder): simple_res_net(
    (net_lin): Linear(in_features=156, out_features=3, bias=True)
    (net_non_lin): feed_forward_nn(
      (net): Sequential(
        (0): Linear(in_features=156, out_features=128, bias=True)
        (1): Tanh()
        (2): Linear(in_features=128, out_features=128, bias=True)
        (3): Tanh()
        (4): Linear(in_features=128, out_features=3, bias=True)
      )
    )
  )
  (fn): simple_res_net(
    (net_lin): Linear(in_features=5, out_features=3, bias=True)
    (net_non_lin): feed_forward_nn(
      (net): Sequential(
        (0): Linear(in_features=5, out_features=128, bias=True)
        (1): Tanh()
        (2): Linear(in_features=128, out_features=128, bias=True)
        (3): Tanh()
        (4): Linear(in_features=128, out_features=3, bias=True)
      )
    )
  )
  (hn): simple_res_net(
    (net_lin): Linear(in_features=3, out_features=2, bias=True)
    (net_non_lin): feed_forward_nn(
    